# K4 · Huấn luyện ma trận {Qwen, LLaMA} × {LoRA fp16, QLoRA nf4} + learning curve (T11)
Cùng dữ liệu, cùng siêu tham số (từ K3), cùng số epoch — chỉ đổi model / lượng tử hóa. Mỗi ô ~45–90 phút trên T4.

**Phiên Kaggle tối đa 12 giờ** → nếu không đủ, chạy notebook nhiều lần, mỗi lần đặt `CELLS` khác nhau (kết quả từng ô được ghi riêng, ô đã xong tự bỏ qua nếu nạp lại thư mục output cũ làm dataset).

In [ ]:
%%capture
!pip install -q "transformers==4.56.2" "trl==0.21.0" "peft==0.17.0" "bitsandbytes==0.47.0" "accelerate>=1.4" "datasets>=3.0" rank-bm25 optuna

In [ ]:
import sys, glob, json, os, time
sys.path.insert(0, glob.glob('/kaggle/input/**/code/src', recursive=True)[0])
from kimochi.kaggle_env import setup, gpu_info
ROOT = setup()
import pandas as pd
from pathlib import Path
RES = ROOT / 'results'
print(gpu_info())

In [ ]:
from kimochi import llm
best_path = RES / 'optuna_best.json'
HP = json.loads(best_path.read_text())['best_params'] if best_path.exists() else {'r': 16, 'lr': 2e-4}
CELLS = [('qwen', 'nf4'), ('qwen', 'fp16'), ('llama', 'nf4'), ('llama', 'fp16')]
SEEDS = [42]        # thêm 7, 1234 nếu còn GPU (mean ± std)
EPOCHS = 2
DATA_DIR = 'splits_v2'   # bộ dữ liệu v2 (mặc định); 'splits' = v1 closed-book
HF_USER = None      # vd 'thanhtrung665' → đẩy adapter lên HF Hub (private)
print(HP)

In [ ]:
results = []
for mk, q in CELLS:
    for seed in SEEDS:
        out = f'/kaggle/working/runs/{mk}_{q}_s{seed}'
        if Path(out, 'result.json').exists():
            results.append(json.loads(Path(out, 'result.json').read_text())); continue
        res = llm.train(ROOT, mk, q, out, seed=seed, lr=HP['lr'], r=HP['r'], epochs=EPOCHS, data_dir=DATA_DIR)
        if HF_USER:
            res['hub'] = llm.push_adapter(res['adapter'], f'{HF_USER}/kimochi-{mk}-{q}-s{seed}')
        results.append(res); print(res)
        pd.DataFrame(results).to_csv(RES / 'train_matrix.csv', index=False)
pd.DataFrame(results)

## Ablation dữ liệu v1 ↔ v2 (cùng model, cùng siêu tham số) — chứng minh tác dụng của làm sạch, RAFT, mẫu công cụ

In [ ]:
out = '/kaggle/working/runs/qwen_nf4_s42_v1data'
if not Path(out, 'result.json').exists():
    r1 = llm.train(ROOT, 'qwen', 'nf4', out, lr=HP['lr'], r=HP['r'], epochs=EPOCHS, data_dir='splits')
    pd.DataFrame([r1]).to_csv(RES / 'train_v1data.csv', index=False)
print(json.loads(Path(out, 'result.json').read_text()))

## Đường cong loss (phát hiện quá khớp)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(CELLS), figsize=(4 * len(CELLS), 3.2), sharey=True)
for ax, (mk, q) in zip(axes, CELLS):
    h = json.loads(Path(f'/kaggle/working/runs/{mk}_{q}_s{SEEDS[0]}/history.json').read_text())
    tr = [(x['step'], x['loss']) for x in h if 'loss' in x]; ev = [(x['step'], x['eval_loss']) for x in h if 'eval_loss' in x]
    ax.plot(*zip(*tr), label='train'); ax.plot(*zip(*ev), 'o-', label='val'); ax.set_title(f'{mk} {q}'); ax.set_xlabel('bước')
axes[0].set_ylabel('loss'); axes[0].legend(); plt.tight_layout(); plt.savefig(RES / 'loss_curves.png', dpi=200)
for (mk, q) in CELLS:
    Path(RES / 'histories').mkdir(exist_ok=True)
    Path(RES / 'histories' / f'{mk}_{q}.json').write_text(Path(f'/kaggle/working/runs/{mk}_{q}_s{SEEDS[0]}/history.json').read_text())

## Learning curve theo cỡ dữ liệu (mô hình tốt nhất) — trả lời 'có đáng tạo thêm dữ liệu không'

In [ ]:
tm = pd.DataFrame(results)
best = tm.sort_values('eval_loss').iloc[0]; print('Ô tốt nhất:', best.model, best.quant)
lc = []
for frac in [0.25, 0.5]:
    out = f'/kaggle/working/runs/lc_{best.model}_{best.quant}_{frac}'
    r = llm.train(ROOT, best.model, best.quant, out, lr=HP['lr'], r=HP['r'], epochs=EPOCHS, data_frac=frac); lc.append(r)
lc.append({**best.to_dict(), 'data_frac': 1.0})
lc = pd.DataFrame(lc); lc.to_csv(RES / 'learning_curve.csv', index=False)
plt.figure(figsize=(5, 3)); plt.plot(lc.data_frac * 100, lc.eval_loss, 'o-'); plt.xlabel('% dữ liệu train'); plt.ylabel('val loss')
plt.savefig(RES / 'learning_curve.png', dpi=200, bbox_inches='tight'); lc[['data_frac', 'eval_loss', 'train_minutes']]

In [ ]:
!cd /kaggle/working && zip -qr adapters.zip runs -x '*/checkpoint-*'  # tải về nếu không dùng HF Hub
!ls -la /kaggle/working